In [9]:
import numpy as np
import json
import requests
from sklearn.linear_model import LogisticRegression
import os

# Replace <EVALUATOR_IP> and <PORT> with the correct values
evaluator_base_url = "http://83.136.253.59:30139"
# Example: evaluator_base_url = "http://127.0.0.1:5000"


In [10]:
import numpy as np

def flip_labels(X, y, percent_poisoned):
    np.random.seed(42)  # for reproducibility
    y_poisoned = y.copy()
    n_samples = len(y)

    # Pick 60% of indices randomly
    num_to_flip = int(percent_poisoned * n_samples)
    flip_indices = np.random.choice(n_samples, size=num_to_flip, replace=False)

    # Check number of classes
    unique_classes = np.unique(y)

    if len(unique_classes) == 2:
        # Binary classification: flip 0 <-> 1
        y_poisoned[flip_indices] = 1 - y_poisoned[flip_indices]
    else:
        # Multiclass: rotate label (e.g., 0 -> 1, 1 -> 2, ..., n-1 -> 0)
        for idx in flip_indices:
            current = y_poisoned[idx]
            available = [c for c in unique_classes if c != current]
            y_poisoned[idx] = np.random.choice(available)

    return X, y_poisoned


In [18]:
# Ensure the flip_labels function from cell 48xGBdsKZRvA is the one being used.
# If there are multiple flip_labels definitions, you might need to adjust imports or cell order.

X_poisoned, y_poisoned = flip_labels(X_train, y_train, percent_poisoned=0.60)

# Now you can proceed with using X_poisoned and y_poisoned
print("Labels flipped successfully.")
print(f"Shape of X_poisoned: {X_poisoned.shape}")
print(f"Shape of y_poisoned: {y_poisoned.shape}")

NameError: name 'X_poisoned' is not defined

In [19]:
import numpy as np

def flip_labels(X, y, percent_poisoned):
    np.random.seed(42)  # Ensure reproducibility
    y_poisoned = y.copy()
    n = len(y)

    # Number of samples to flip
    num_to_flip = int(percent_poisoned * n)

    # Randomly pick indices to flip
    indices = np.random.choice(n, size=num_to_flip, replace=False)

    # Flip labels (assumes binary: 0 ↔ 1)
    y_poisoned[indices] = 1 - y_poisoned[indices]

    return X, y_poisoned


In [ ]:
dataset_filename = "/label_flipping_dataset.npz"

try:
    data = np.load(dataset_filename)
    X_train = data["Xtr"]
    y_train = data["ytr"]
    X_test = data["Xte"]
    y_test = data["yte"]
    print("Data loaded successfully from single .npz file.")
    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")
    data.close()
except FileNotFoundError:
    print(f"Error: Dataset file '{dataset_filename}' not found.")
    print("Make sure the .npz data file is in the correct directory.")
    raise
except KeyError as e:
    print(f"Error: Could not find expected array key '{e}' in the .npz file.")
    raise

In [ ]:
# Implement your attack code in this stub
def flip_labels(y, poison_percentage, seed):
    """
    Flips a percentage of labels in a given label array for binary classification (0 <-> 1).

    Args:
        y (np.ndarray): The original label array.
        poison_percentage (float): The percentage of labels to flip (e.g., 0.60 for 60%).
        seed (int): The random seed for reproducibility.

    Returns:
        tuple: A tuple containing:
            - np.ndarray: The poisoned label array.
            - np.ndarray: An array of the indices that were flipped.
    """
    np.random.seed(seed)
    y_poisoned = y.copy()
    n_samples = len(y)

    # Calculate the number of labels to flip
    num_to_flip = int(poison_percentage * n_samples)

    # Randomly select indices to flip
    flipped_idx = np.random.choice(n_samples, size=num_to_flip, replace=False)

    # Flip labels (assuming binary: 0 <-> 1)
    # Ensure labels are treated as integers for flipping
    y_poisoned[flipped_idx] = 1 - y_poisoned[flipped_idx].astype(int)

    return y_poisoned, flipped_idx


# ------------------------------------------------------------------------
# --- The rest is templated and you should not need to change anything ---
# ------------------------------------------------------------------------
poison_rate = 0.60
random_seed = 1337
y_train_poisoned, flipped_idx = flip_labels(y_train, poison_rate, random_seed)

print(f"Shape of poisoned labels: {y_train_poisoned.shape}")
print(f"Number of labels flipped: {len(flipped_idx)}")
print(f"Original labels at flipped indices (first 5): {y_train[flipped_idx[:5]]}")
print(
    f"Poisoned labels at flipped indices (first 5): {y_train_poisoned[flipped_idx[:5]]}"
)

In [ ]:
model = LogisticRegression(random_state=random_seed)
model.fit(X_train, y_train_poisoned)
print("Model trained successfully on poisoned data.")
weights = model.coef_
intercept = model.intercept_
print(f"Extracted weights (shape): {weights.shape}")
print(f"Extracted intercept (shape): {intercept.shape}")

In [ ]:
health_check_url = f"{evaluator_base_url}/health"
print(f"Checking evaluator health at: {health_check_url}")
if "<EVALUATOR_IP>" in evaluator_base_url:
    print("\n--- WARNING ---")
    print(
        "Please update the 'evaluator_base_url' variable with the correct IP and Port before running!"
    )
    print("-------------")
else:
    try:
        response = requests.get(health_check_url, timeout=10)
        response.raise_for_status()
        health_status = response.json()
        print("\n--- Health Check Response ---")
        print(f"Status: {health_status.get('status', 'N/A')}")
        print(f"Message: {health_status.get('message', 'No message received.')}")
        if health_status.get("status") != "healthy":
            print(
                "\nWarning: Evaluator service reported an unhealthy status. It might still be starting up or encountered an issue (like loading data)."
            )
    except requests.exceptions.ConnectionError as e:
        print(f"\nConnection Error: Could not connect to {health_check_url}.")
        print("Please check:")
        print("  1. The evaluator URL (IP address and port) is correct.")
        print("  2. The evaluator Docker container is running.")
        print(
            "  3. There are no network issues (firewalls, etc.) blocking the connection."
        )
    except requests.exceptions.Timeout:
        print(f"\nTimeout Error: The request to {health_check_url} timed out.")
        print(
            "The server might be taking too long to respond or there could be network issues."
        )
    except requests.exceptions.RequestException as e:
        print(f"\nError during health check request: {e}")
        print("Check the URL format and ensure the server is running.")
    except json.JSONDecodeError:
        print("\nError: Could not decode JSON response from health check.")
        print("The server might have sent an invalid response.")
        print(
            f"Raw response status: {response.status_code}, Raw response text: {response.text}"
        )
    except Exception as e:
        print(f"\nAn unexpected error occurred during health check: {e}")

In [ ]:
evaluator_url = f"{evaluator_base_url}/evaluate"
payload = {"weights": weights.tolist(), "intercept": intercept.tolist()}
print(f"Attempting submission to: {evaluator_url}")
if "<EVALUATOR_IP>" in evaluator_base_url:
    print("\n--- WARNING ---")
    print(
        "Please update the 'evaluator_base_url' variable with the correct IP address and Port before running this cell!"
    )
    print("-------------")
else:
    print(f"Payload: {json.dumps(payload)}")
    try:
        response = requests.post(evaluator_url, json=payload, timeout=30)
        response.raise_for_status()
        result = response.json()
        print("\n--- Evaluator Response ---")
        if result.get("success"):
            print("Attack Successful!")
            print(f"Accuracy evaluated by server: {result.get('accuracy'):.4f}")
            print(f"Flag: {result.get('flag')}")
        else:
            print("Evaluation Failed.")
            accuracy_val = result.get("accuracy")
            accuracy_str = f"{accuracy_val:.4f}" if accuracy_val is not None else "N/A"
            print(f"Accuracy evaluated by server: {accuracy_str}")
            print(f"Message: {result.get('message')}")
            print(
                "Hints: Did you poison exactly 60% of the data? Did you use the seed 1337 for flipping labels?"
            )
    except requests.exceptions.ConnectionError as e:
        print(
            f"\nConnection Error: Could not connect to the evaluator API at {evaluator_url}."
        )
        print("Please check:")
        print("  1. The evaluator URL (IP address and port) is correct.")
        print("  2. The evaluator Docker instance is spawned.")
        print(
            "  3. There are no network issues (firewalls, etc.) blocking the connection."
        )
    except requests.exceptions.Timeout:
        print(f"\nTimeout Error: The request to {evaluator_url} timed out.")
        print("The server might be slow, or there could be network issues.")
    except requests.exceptions.RequestException as e:
        print(f"\nError connecting to evaluator API: {e}")
        print("Please check the evaluator URL and ensure the instance is spawned.")
    except json.JSONDecodeError:
        print("\nError decoding JSON response from the evaluator.")
        print("The server might have sent an invalid response.")
        print(
            f"Raw response status: {response.status_code}, Raw response text: {response.text}"
        )
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")